# Open-Ended Evolution of Self-Improving Agents

> The first six lectures share one aim: a person designs a better **Agent**. Lecture 4 builds the ReAct loop, lecture 5 designs task decomposition and tree search, and lecture 6 trains the Agent's parameters with **reinforcement learning**. The designer is always a person; the Agent is only the object being designed.
>
> This lecture hands the designer role to search: one Agent designs another **Agent**—generating candidate code, scoring it on tasks, storing good results in an archive, then drawing on those results to generate the next batch. This cycle of mutation, selection, and accumulation is **open-ended evolution**. We start from ADAS's minimal closed loop, scale it to AI Scientist's research workflow, then watch AlphaEvolve evolve entire source files as genomes, and finally discuss failure modes.

Start with a minimal example. An Agent that only "computes directly" on two-digit multiplication, such as 23×47, has single-shot accuracy 0.55—more than half the problems are wrong.

Let another program read its code and edit it. First change: replace "compute directly" with "decompose, then compute" (compute 23×40, then 23×7, then add). Re-evaluate: accuracy rises from 0.55 to 0.80.

Second change: after decomposing, add a "check" step (recompute in another order; if the results disagree, try again). Accuracy rises further to 0.93.

Edit, run, keep the higher score. After a few rounds the Agent is stronger. This loop—a program editing Agent code so that later versions score higher—is the subject of this lecture: **open-ended evolution**.

It differs from lecture 6. Training changes model parameters; once training ends, ability is fixed. Open-ended evolution changes a higher layer: the Agent's code, research ideas, or algorithm structure. This lecture uses three real systems: ADAS evolves Agent code, AI Scientist evolves research ideas, and AlphaEvolve evolves entire source files.

The loop also has a risk. When the system scores its own outputs, and those scores drift from the true objective, it hunts for shortcuts—scores look high, the task is not done. That failure is **reward hacking**. Independent evaluation is the only guardrail.

The next section starts with the basic step: how one program edits another Agent's code.

In open-ended evolution, the program that edits Agents is a **meta-Agent**. This section builds a minimal system: the meta-Agent reads existing Agent code, writes new code, evaluates it on tasks, and keeps the better versions. The system is **ADAS** (Automated Design of Agentic Systems). It treats "design an Agent" as a search problem.

First, fix what an Agent looks like. ADAS packs every component—prompts, tool calls, control flow—into one function, `forward(task)`. How that function is written is how the Agent acts, so "search for a better Agent" means "search over ways to write forward".

There are too many ways to write the code for a person to try them one by one. ADAS searches with a loop that turns on three pieces: **mutation** produces new code (a language model reads old code and writes new code), **selection** judges quality (an evaluation function runs on a fixed task set and returns accuracy), and a **population** keeps the good versions (an archive dictionary storing name, code, and score, used as context for the next round).

This section starts with two preparations: fixing the task and the evaluation function.

Designing an Agent has two levels. The low level tunes parameters: wording in a prompt, number of reasoning steps, number of samples. The high level changes structure: a different decision procedure, or even a different control flow. ADAS searches the high level—its search space is the Agent's source code, not a set of parameters.

Code can be a search space because Python is expressive enough: conditionals, loops, recursion, calls to other functions. Any computable logic can be written. That "anything computable can be expressed" property is called **Turing completeness**. The space is large enough that designs a person never considered may sit inside it; the cost of that size is that search has to be cyclic.

A minimal example shows how code carries a strategy. The baseline Agent's forward is one line:

```text
def forward(task):
    a, b = task
    return solve(a, b, recipe='direct')
```

The value of recipe selects the strategy: `direct` computes in one shot, `decompose` splits then computes, `ensemble` runs several recipes and votes. Change one word, and the behavior changes. In a real system, forward can also contain `if`, `for`, and repeated calls; richer control flow expresses more strategies.

For that space to be searchable, two more pieces are required: a fixed task set, and a ruler that scores candidates (the evaluation function). The next cells fix both.

In [ ]:
# Task: two-digit multiplication. The framework provides solve(a, b, recipe),
# which computes the answer by a "reasoning recipe";
# the environment uses deterministic pseudo-randomness to decide which
# problems each recipe fails on, simulating an underlying model's
# error rate at different reasoning depths. np.random.seed keeps later demos reproducible.
import numpy as np
np.random.seed(42)

A = np.repeat(np.arange(11, 16), 8)   # 5 tens digits, each repeated 8 times
B = np.tile(np.arange(11, 19), 5)     # 8 units digits, tiled 5 times
PROBLEMS = list(zip(A.tolist(), B.tolist()))
TRUE = [a * b for a, b in PROBLEMS]

RECIPE_RATE = {"direct": 0.40, "decompose": 0.12,
               "decompose_check": 0.05, "ensemble": 0.00}

def is_tricky(a, b, recipe):
    """Whether this recipe fails on this problem: a seed-determined deterministic check."""
    r = np.random.RandomState(1000 * (a % 10) + (b % 10) + len(recipe))
    return r.rand() < RECIPE_RATE[recipe]

def solve(a, b, recipe):
    """Base solver: compute a*b by recipe; on a miss, swap tens and units digits."""
    if is_tricky(a, b, recipe):
        return str((a % 10) * (b % 10) + 100 * (a // 10) * (b // 10))
    return str(a * b)

for rec in RECIPE_RATE:
    acc = sum(1 for (a, b), t in zip(PROBLEMS, TRUE)
              if solve(a, b, rec) == str(t)) / len(PROBLEMS)
    print(f"recipe {rec:<14} set error rate {RECIPE_RATE[rec]:.2f} measured accuracy {acc:.2f}")

The cell above reports accuracy for four recipes, which is exactly how we judge each design. Using that number as the quality of a candidate is the score that selection needs. That score has a name: **fitness**. The results as a table:

| Recipe | Set error rate | Measured accuracy on 40 problems |
|:---|:---|:---|
| direct | 0.40 | 0.55 |
| decompose | 0.12 | 0.80 |
| decompose_check | 0.05 | 0.93 |
| ensemble | 0.00 | 1.00 |

Higher error rate, lower accuracy; ensemble with error rate 0 misses none of the problems. Now locate the three components of open-ended evolution in this code.

The mutation operator produces new variants: it reads existing Agent code and writes a new `forward`. In this section a language model plays that role. The selection mechanism judges which variant is better: an evaluation function that runs the variant on a fixed task set and returns accuracy, and that accuracy is fitness. The population stores variants and scores so that new variants are not invented from nothing, but can refer to earlier results; in this section the population is the archive dictionary.

All three are required. Without mutation we have only existing Agents and the loop cannot advance. Without selection there is no distinction between good and bad, so search has no direction. Without an archive every round starts from zero and nothing accumulates. Mutation explores, selection judges, the archive remembers; together they form a loop that can improve itself.

The next cell shows mutation in its smallest form: change one word, and the score changes.

This subsection shows how a mutation operator produces new variants. In code space, mutation is editing a line of code; in parameter space, mutation is changing a parameter value. We start with parameter-level mutation: replace one word in the Agent's prompt, and the score follows. In a real system that score comes from actually running the Agent; here a synthetic deterministic scoring function stands in, so the effect of mutation is easy to watch. In this synthetic function, each action keyword adds a fixed accuracy bonus.

In [ ]:
# Minimal mutation-operator demo: insert an action keyword into the prompt, and the score changes.
# The scoring function is deterministic: base accuracy 0.30, plus bonuses for matched keywords.
KEYWORD_BONUS = {"decompose": 0.15, "check": 0.15, "estimate": 0.10, "stepwise": 0.05}

def prompt_score(prompt):
    """Prompt score: base accuracy 0.30, plus bonuses for matched keywords."""
    score = 0.30
    for word, bonus in KEYWORD_BONUS.items():
        if word in prompt:
            score += bonus
    return score

CANDIDATES = ["decompose", "check", "estimate", "stepwise"]

def mutate_prompt(prompt, seed):
    """Pick one candidate word at random and append it to the prompt, completing one mutation."""
    rng = np.random.RandomState(seed)
    word = CANDIDATES[int(rng.randint(len(CANDIDATES)))]
    return prompt + ", " + word

base = "Please compute this multiplication"
for k in range(4):
    child = mutate_prompt(base, k)
    print(f"mutation {k + 1}: \"{child}\" score {prompt_score(child):.2f}")

print("original prompt score:", round(prompt_score(base), 2))

In the run, the original prompt scores 0.30; after four mutations the scores are 0.45, 0.45, 0.45, and 0.40. Each candidate word has a fixed bonus: decompose +0.15, check +0.15, estimate +0.10, stepwise +0.05. The scoring function is deterministic, so the same prompt always gets the same score, which lets us watch the causal path from mutation to score without noise.

This scoring function is synthetic: it is built so that matching a keyword adds a bonus. In a real system the score comes from actually running: give this prompt to an Agent, run it on the 40 problems, and use accuracy as the score. The synthetic score is useful because a first-time reader sees only the mechanism—change one word, watch how much the score moves.

That a one-word change moves the score matters, because evolution navigates by score differences. If no edit affected the score, mutation would be pointless and selection would have nothing to work with. If edits can raise or lower the score, the mutation operator has a signal: higher-scoring mutations are worth keeping, lower-scoring ones can be dropped. This round we only observe; we do not select. The scoring function fixes the mutation-to-score map; the next subsection on parameter-space search folds selection into the loop.

This subsection folds selection into the loop. Given a scoring function, search finds high-scoring parameters as follows. Mutation produces new variants; selection is delegated to the scoring function: high-scoring variants are kept, low-scoring ones are dropped. We treat the Agent's three parameters—reasoning steps, ensemble samples, and number of tools—as a three-dimensional point $p=(n_{cot}, n_{samples}, n_{tools})$. The scoring function returns a deterministic score for each point.

$$score(p) = 0.20 + 0.30·tanh(n_{cot}/3) + 0.10·tanh(n_{samples}/4) + 0.05·tanh((n_{tools}-2)/1.5) - 0.03·max(0, n_{tools}-5)$$

The formula estimates "given this parameter vector, how well the Agent can do". The first three terms give diminishing-returns gains from reasoning steps and ensemble samples; the last term penalizes using too many tools. tanh squeezes its input into $(-1, 1)$, so larger parameters approach a ceiling—that is diminishing returns. The function is synthetic and exists only so we can watch the search mechanism; in a real system the score comes from actually running.

Two points, computed by hand, show the range of scores. $p=(0,1,2)$: $tanh(0)=0$, $tanh(0.25)≈0.245$, score $≈0.20+0.0245≈0.22$. $p=(6,8,3)$: $tanh(2)≈0.964$, $tanh(2/3)≈0.583$, score $≈0.20+0.30×0.964+0.10×0.964+0.05×0.583≈0.61$. The next cells check these two points in code, then run hill climbing and random search.

In [ ]:
def agent_score(p):
    """Deterministic score of three parameters (reasoning steps, ensemble samples, tools)."""
    n_cot, n_samples, n_tools = p
    acc = 0.20 + 0.30 * np.tanh(n_cot / 3.0)
    acc += 0.10 * np.tanh(n_samples / 4.0)
    acc += 0.05 * np.tanh((n_tools - 2) / 1.5)
    acc -= 0.03 * max(0, n_tools - 5)
    return float(np.clip(acc, 0.0, 1.0))

for p in [(0, 1, 2), (6, 8, 3)]:
    print(f"agent_score{p} = {agent_score(p):.4f}")

def neighbors(p):
    """Return four neighboring parameter points: reasoning steps and ensemble samples each plus or minus one."""
    n_cot, n_samples, n_tools = p
    return [(n_cot + 1, n_samples, n_tools),
            (n_cot, n_samples + 1, n_tools),
            (max(0, n_cot - 1), n_samples, n_tools),
            (n_cot, max(1, n_samples - 1), n_tools)]

In [ ]:
def hill_climb(start, steps=12):
    """From start, move each step to the best higher-scoring neighbor; return the trace."""
    p = start
    trace = [(p, agent_score(p))]
    for _ in range(steps):
        best_c = max(neighbors(p), key=agent_score)
        if agent_score(best_c) <= agent_score(p):
            break
        p = best_c
        trace.append((p, agent_score(p)))
    return trace

def random_search(budget=30, seed=0):
    """Sample budget independent parameter points; record the best score so far."""
    rng = np.random.RandomState(seed)
    best = -1.0
    trace = []
    for _ in range(budget):
        p = (int(rng.randint(0, 10)), int(rng.randint(1, 12)),
             int(rng.randint(0, 8)))
        s = agent_score(p)
        if s > best:
            best, best_p = s, p
        trace.append(best)
    return best_p, trace

hc_trace = hill_climb((0, 1, 2))
best_rs, rs_trace = random_search(30, seed=2)

print("hill-climbing trace (params, score):")
for p, s in hc_trace:
    print(f"  {p}  {s:.3f}")
print("hill-climbing final score:", round(hc_trace[-1][1], 3))
print("random-search best params:", best_rs, "score:", round(agent_score(best_rs), 3))

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

_, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].plot([s for _, s in hc_trace], "o-", color="#2c7fb8")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Score")
axes[0].set_title("Hill climbing")
axes[1].plot(rs_trace, ".-", color="#d95f0e")
axes[1].set_xlabel("Sample")
axes[1].set_ylabel("Best score so far")
axes[1].set_title("Random search")
plt.tight_layout()
plt.show()
print("hill climbing from", round(hc_trace[0][1], 3), "to", round(hc_trace[-1][1], 3))

The two curves show the difference between two search strategies. Hill climbing starts at (0, 1, 2), each step moves to the highest-scoring of four neighbors, and it stops at (7, 6, 2) with score 0.585. Random search samples 30 independent parameter points and records the best score so far; its best visit is (6, 10, 5) with score 0.636.

Read the two results against each other. Hill climbing moves up every step, the path is continuous, and the endpoint 0.585 is much better than the start 0.224; but it can see only the four neighbors that touch the current point, so its view is local. The scoring function may have more than one peak in parameter space, and hill climbing's endpoint need not be the highest. Random search does not walk along neighbors; it scatters points. With enough samples it is more likely to land in a higher-scoring region, and in this example 0.636 beats the hill-climbing endpoint.

The contrast makes one point: local repair alone, or blind trial alone, is crude search. To take Agent design past what a person can invent by hand, mutation, selection, and memory have to be combined into a loop. The next section enters ADAS's minimal closed loop.

The previous two subsections demonstrated mutation and selection in parameter space, but ADAS's real search space is code. This subsection combines mutation, selection, and memory into a closed loop and shows how the loop turns. The minimal loop has four steps: the meta-Agent reads an archive summary and writes a `forward` function; we compile it into a callable; we evaluate accuracy on the arithmetic task; we store the result, including the code, in the archive. The archive starts from two baseline Agents and grows round by round. First, the infrastructure.

In [ ]:
import sys, os
_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, 'llm_client.py')):
    _root = os.path.dirname(_root)
    if _root == os.path.dirname(_root):
        break
if _root not in sys.path:
    sys.path.insert(0, _root)
from llm_client import get_llm

client = get_llm()
print("scripted demo:", False)

In [ ]:
def wrap_recipe(recipe):
    """Wrap a reasoning recipe as a forward source-code string."""
    return f"""def forward(task):
    a, b = task
    return solve(a, b, recipe='{recipe}')"""

def compile_agent(src):
    """Compile forward source into a callable; framework functions are visible to generated code."""
    ns = globals().copy()
    exec(src, ns)
    return ns["forward"]

def eval_agent(fn, problems=PROBLEMS, true=TRUE):
    """Run the Agent on the task set and return accuracy."""
    correct = sum(1 for (a, b), t in zip(problems, true)
                  if fn((a, b)) == str(t))
    return correct / len(problems)

# archive: dict storing "name → (source, score)". Start from two baselines, then compute true scores.
archive = {"baseline_direct": (wrap_recipe("direct"), 0.0),
           "baseline_decompose": (wrap_recipe("decompose"), 0.0)}
for name, (src, _) in archive.items():
    archive[name] = (src, eval_agent(compile_agent(src)))
for name, (_, acc) in archive.items():
    print(f"{name}: accuracy {acc:.2f}")

In [ ]:
def extract_recipe(src):
    """Read the recipe name from source; return 'unknown' if parsing fails."""
    for rec in RECIPE_RATE:
        if f"recipe='{rec}'" in src:
            return rec
    return "unknown"

def meta_prompt(archive):
    """Build the meta-Agent prompt: read an archive summary, write new forward code."""
    lines = []
    for name, (src, acc) in archive.items():
        lines.append("- " + name + ": accuracy " + f"{acc:.2f}"
                     + ", recipe " + extract_recipe(src))
    summary = "\n".join(lines)
    return ("You are a meta-Agent, responsible for designing an Agent that solves "
            "two-digit multiplication. The framework provides "
            "solve(a, b, recipe); recipe may be direct / decompose / "
            "decompose_check / ensemble. Existing Agents:\n" + summary +
            "\nWrite a new forward(task) function, wrapped in ```python.")

def generate_agent(client, archive, it):
    """The meta-Agent generates one Agent's code. The scripted path returns a placeholder."""
    if False:
        # Scripted placeholder: rotate through a recipe pool to show stepping-stone behavior
        pool = ["decompose_check", "ensemble", "decompose", "decompose_check"]
        return wrap_recipe(pool[it % len(pool)])
    reply = client.chat([{"role": "user", "content": meta_prompt(archive)}])
    code = reply.split("```python")[-1].split("```")[0].strip()
    if "def forward" not in code:
        code = wrap_recipe("decompose_check")
    return code

print(meta_prompt(archive))
print("---- meta-Agent output (scripted placeholder on the scripted path) ----")
print(generate_agent(client, archive, 0))

In [ ]:
def adas_loop(client, archive, rounds=4):
    """Meta-Agent search: generate → compile → evaluate → store. Return the per-round log."""
    log = []
    for it in range(rounds):
        src = generate_agent(client, archive, it)
        try:
            acc = eval_agent(compile_agent(src))
        except Exception:
            acc = 0.0
        name = "discovered_" + str(it)
        best = max(s for _, s in archive.values())
        added = acc > best
        if added:
            archive[name] = (src, acc)
        log.append((it, extract_recipe(src), acc, added))
    return log

log = adas_loop(client, archive, rounds=4)
for it, recipe, acc, added in log:
    flag = "keep" if added else "drop"
    print(f"round {it}: recipe {recipe:<14} accuracy {acc:.2f}  {flag}")

print("final archive (stepping-stone record):")
for name, (src, acc) in archive.items():
    print(f"  {name:<18} recipe {extract_recipe(src):<14} accuracy {acc:.2f}")

Replay the output round by round, and watch the input and output of each of the four steps in the closed loop.

The start is the two baselines in the archive: baseline_direct at accuracy 0.55, baseline_decompose at accuracy 0.80. That is the initial contents of memory, and the entire summary the meta-Agent can read in round 0.

Round 0, four steps:
1. Mutation. The meta-Agent reads the archive summary (two lines: direct 0.55, decompose 0.80) and writes a new forward: `return solve(a, b, recipe='decompose_check')`.
2. Compile. `exec` turns that source into a callable forward.
3. Evaluate. Run on the 40 problems; accuracy 0.93.
4. Store. Current historical best = max(0.55, 0.80) = 0.80. 0.93 is greater than 0.80, so it is stored as discovered_0.

Round 1: the archive now has three summaries. The meta-Agent can see discovered_0's 0.93 and writes ensemble. Evaluation 1.00; best has already risen to 0.93; 1.00 is greater than 0.93, so it is stored as discovered_1.

Round 2: the meta-Agent writes decompose. 0.80 does not exceed the current best 1.00, so it is dropped and the archive is unchanged.

Round 3: it writes decompose_check. 0.93 likewise does not exceed 1.00, so it is dropped.

There is only one store condition: `acc > current historical best`. That guarantees the archive records only genuine progress; repeated or regressing candidates are discarded. The final archive stabilizes at four entries: when there is no new breakthrough, the archive stops changing.

The closed loop's key is the last step, storing. Code that enters the archive appears in the next round's prompt summary and directly affects what the meta-Agent writes next. That is improvement from scores across rounds: high-scoring code remains as a new starting point, low-scoring code disappears, and the history the meta-Agent can compose over grows longer.

The archive's value is that it serves as memory for the mutation operator. In round 0 above, the discovered decompose_check borrows decompose from the baseline and adds a check; in round 1, ensemble packages decompose_check into a vote. Each store builds on the previous round: later Agents combine components of earlier Agents rather than starting from zero. The paper calls this stepwise accumulation a **stepping stone**. This section's archive starts from two baselines, records two improvements, then stops growing: with no new breakthrough, the archive no longer changes.

The name stepping stone comes from the river-crossing image: one steps on stones in the water, and each stone is a foothold for the next step. A stepping stone in evolution means the same thing: one solution becomes the material for constructing the next.

The chain in this section's archive:

```text
baseline_decompose (0.80)
        ↓ borrow the decompose idea
discovered_0: decompose_check (0.93)   # decompose plus a check
        ↓ wrap decompose_check as a vote
discovered_1: ensemble (1.00)          # several recipes run together, then vote
```

discovered_0 is not designed from nothing; it adds a check on top of baseline_decompose. discovered_1 then wraps discovered_0 as ensemble. Each step reuses the previous step's result, which is the archive acting as memory. Without an archive, every round the meta-Agent can only design from scratch; with an archive, it can read historically best code and keep climbing the existing steps.

The paper's ARC-challenge data shows the same pattern: round 3 produces multi-path CoT plus revision; only at round 25 are several kinds of feedback assembled into the final Agent. Early discoveries are mostly simple components; later rounds combine them into complex designs. The stepping-stone mechanism is why open-ended evolution is not a fresh search from scratch each time, but a continued climb along historically accumulated steps—and why it is called open-ended.

This section scales the loop from section 1, from designing Agents to doing research. ADAS put Agent design into a loop. AI Scientist scales the same loop to a full research workflow: propose a research idea, write code and run experiments, write the results as a paper, have an automatic reviewer score it, then store accepted ideas together with their scores in a knowledge archive. Compared with ADAS there are three upgrades: variants change from Agent code to research ideas, fitness changes from task accuracy to paper review, and the population changes from a code library to a knowledge base.

The automatic reviewer is the key to the closed loop, and also the risk. The paper uses a GPT-4o reviewer agent that scores by NeurIPS guidelines, reaching near-human judgment on 500 ICLR 2022 papers. When author and reviewer are the same AI, the closed loop has no external ground truth—no objective standard outside the system that can decide whether a result is right. We watch this structure with a minimal loop: a language model proposes an idea, a deterministic environment produces experimental metrics, and a review function decides keep or drop.

In [ ]:
# Experiment environment: from-scratch full-batch gradient descent on fixed regression data.
# An idea changes experimental metrics (validation loss) by editing hyperparameters; everything is deterministic and reproducible.
import numpy as np
np.random.seed(42)

rng = np.random.RandomState(0)
X = rng.randn(80, 8)
w_true = np.array([1.0, 2.0, -1.5, 1.0, -0.5, 0.8, -0.3, 0.6])
y = X @ w_true + 0.6 * rng.randn(80)
Xtr, Xva = X[:60], X[60:]
ytr, yva = y[:60], y[60:]

def fit_linear(Xtr, ytr, Xva, yva, lr=0.05, wd=0.0, epochs=5, momentum=False, optimizer=None, learning_rate=None, **kwargs):
    """Fit a linear model with full-batch gradient descent; return validation mean squared error."""
    if learning_rate is not None:
        lr = learning_rate
    w = np.zeros(8)
    v = np.zeros(8)
    n = len(ytr)
    for _ in range(epochs):
        g = 2.0 * (Xtr.T @ (Xtr @ w - ytr)) / n + 2.0 * wd * w
        v = 0.9 * v + g
        w = w - lr * (v if momentum else g)
    pred = Xva @ w
    return float(np.mean((pred - yva) ** 2))

# Baseline trains only 5 epochs, clearly underfitted, leaving room for ideas to improve
BASE_CFG = {"lr": 0.05, "wd": 0.0, "epochs": 5, "momentum": False}
baseline_val = fit_linear(Xtr, ytr, Xva, yva, **BASE_CFG)
print("baseline config validation loss:", round(baseline_val, 4))

IDEAS = [("Lower the learning rate to 0.005", {"lr": 0.005}),
         ("Add L2 weight decay", {"wd": 0.05}),
         ("Use momentum optimization", {"momentum": True}),
         ("Double the number of training epochs", {"epochs": 10}),
         ("Quadruple the number of training epochs", {"epochs": 20})]
for desc, cfg in IDEAS:
    val = fit_linear(Xtr, ytr, Xva, yva, **dict(BASE_CFG, **cfg))
    print(f"{desc}: validation loss {val:.4f}")

In [ ]:
# Idea generation and review both go through the language model: the scripted path returns placeholders.
def generate_idea(client, it):
    """Propose a research idea; return (description, hyperparameter change)."""
    if False:
        # Scripted placeholder: rotate through the candidate pool
        return IDEAS[it % len(IDEAS)]
    prompt = ("You are a research assistant. The baseline fits linear regression "
              "with 5 epochs of full-batch gradient descent, "
              "validation loss " + f"{baseline_val:.4f}" + ". Propose an idea "
              "that improves the hyperparameter configuration, in two lines:\n"
              "idea: <one sentence>\ncfg: <python dict>")
    reply = client.chat([{"role": "user", "content": prompt}])
    return parse_idea(reply) or IDEAS[it % len(IDEAS)]

def parse_idea(reply):
    """Parse (description, hyperparameter dict) from a reply; return None if parsing fails."""
    desc, cfg = None, None
    for ln in reply.splitlines():
        if ln.startswith("idea:"):
            desc = ln.split(":", 1)[-1].strip()
        if ln.startswith("cfg:"):
            cfg = ln.split(":", 1)[-1].strip()
    if desc is None or cfg is None:
        return None
    try:
        cfg = eval(cfg)
    except Exception:
        return None
    return (desc, cfg) if isinstance(cfg, dict) else None

def parse_scores(reply):
    """Parse a score dict from a review reply; return None if parsing fails."""
    out = {}
    for token in reply.replace(",", " ").split():
        if ":" in token:
            k, v = token.split(":", 1)
        elif "=" in token:
            k, v = token.split("=", 1)
        else:
            continue
        if k in ("soundness", "presentation", "contribution", "overall"):
            try:
                out[k] = float(v)
            except ValueError:
                pass
    return out if len(out) == 4 else None

def review_idea(client, desc, val, seen, baseline):
    """Review one idea; return (score dict, whether accepted).

    The scripted path scores by rule: more loss improvement yields a higher score; duplicate ideas are rejected.
    """
    improve = baseline - val
    if True:
        prompt = ("Review a research idea. Description: " + desc + "\nvalidation loss: "
                  + f"{val:.4f}, baseline: " + f"{baseline:.4f}.\n"
                  "Score it by the NeurIPS guidelines: soundness, presentation, "
                  "contribution, overall (1-10).")
        reply = client.chat([{"role": "user", "content": prompt}])
        parsed = parse_scores(reply)
        if parsed:
            parsed["accept"] = (desc not in seen and parsed["overall"] >= 6
                                and parsed["contribution"] >= 3)
            return parsed
    soundness = float(np.clip(3 + 50 * improve, 1, 10))
    contribution = float(np.clip(1 + 40 * improve, 1, 10))
    novelty = 0.3 if desc in seen else 0.9
    overall = float(np.clip(0.5 * soundness + 0.3 * contribution
                            + 0.2 * novelty, 1, 10))
    accept = desc not in seen and overall >= 6 and contribution >= 3
    return {"soundness": round(soundness, 1), "presentation": 5.0,
            "contribution": round(contribution, 1), "overall": round(overall, 1),
            "accept": accept}

demo = review_idea(client, "Use momentum optimization", baseline_val / 3, set(), baseline_val)
print("review example:", demo)

The reviewer's scores come from a deterministic formula, which is easy to compute by hand. First compute the improvement: improve = baseline validation loss minus this idea's validation loss. Larger improve means the idea helped more. Then map improve onto three score dimensions:

```text
soundness    = clip(3 + 50×improve, 1, 10)     # how reliable the result is
contribution = clip(1 + 40×improve, 1, 10)     # how large the contribution is
novelty      = 0.9 (first-seen idea) or 0.3 (duplicate idea)
overall      = clip(0.5×soundness + 0.3×contribution + 0.2×novelty, 1, 10)
```

The more validation loss drops, the larger improve, and the higher soundness and contribution. When an idea makes loss worse, improve is negative and both scores are clipped to the floor of 1. Clip means a value outside the interval is pushed onto the boundary; here the interval is 1 to 10. Novelty is an independent dimension: a repeated idea is pushed down to 0.3.

Hand calculation of the review example. In the demo, val = baseline ÷ 3 = 2.1295 ÷ 3 ≈ 0.710, improve = 2.1295 − 0.710 ≈ 1.420. Substitute:

```text
soundness    = clip(3 + 50×1.420, 1, 10) = clip(74, 1, 10) = 10
contribution = clip(1 + 40×1.420, 1, 10) = clip(58, 1, 10) = 10
novelty      = 0.9
overall      = clip(0.5×10 + 0.3×10 + 0.2×0.9, 1, 10) = 8.18 ≈ 8.2
```

This matches the output {soundness: 10.0, contribution: 10.0, overall: 8.2, accept: True}. The accept threshold is overall ≥ 6 and contribution ≥ 3: overall reflects overall quality, and contribution is a separate gate so that a well-written idea with no contribution cannot pass.

The three dimensions correspond to three questions in paper review: whether the conclusion is credible, whether there is something new for the field, and whether the work is duplicate effort. In the real AI Scientist the reviewer reads a full paper written by an AI; here that is compressed into a set of numbers. The mechanism is the same.

In [ ]:
def scientist_loop(client, rounds=7):
    """idea → experiment → review → store; return the per-round log and the knowledge archive."""
    archive = {}
    seen = set()
    log = []
    if False:
        print("ideas are scripted LLM placeholders; review scores by rule (placeholder output)")
    for it in range(rounds):
        desc, cfg = generate_idea(client, it)
        val = fit_linear(Xtr, ytr, Xva, yva, **dict(BASE_CFG, **cfg))
        review = review_idea(client, desc, val, seen, baseline_val)
        dup = desc in seen
        if review["accept"]:
            archive[desc] = {"cfg": cfg, "val": val, "review": review}
        seen.add(desc)
        log.append((it, desc, val, review["overall"], review["accept"], dup))
    return log, archive

log, archive = scientist_loop(client, rounds=7)
for it, desc, val, overall, acc, dup in log:
    if dup:
        reason = "drop (duplicate idea)"
    else:
        reason = "keep" if acc else "drop (score too low)"
    print(f"round {it}  {desc:<36} val={val:.4f} overall={overall}  {reason}")
print("knowledge archive (accepted ideas):")
for desc, info in archive.items():
    print(f"  {desc}: val={info['val']:.4f} overall={info['review']['overall']}")

Replay 7 rounds and watch the closed loop of idea, experiment, review, and store. The baseline validation loss is 2.1295. Each idea first edits hyperparameters, then runs the experiment to get a new validation loss, then goes to review.

The first two rounds are negative examples. Round 0's idea lowers the learning rate to 0.005; validation loss rises to 4.7076. improve is negative, soundness is clipped to 1, overall is only 1.0, which does not reach 6, so it is dropped. Round 1 adds L2 weight decay; loss 2.1516, slightly worse than baseline, overall 1.4, also dropped. Ideas that worsen validation loss do not enter the archive.

Round 2 uses momentum optimization and drops loss to 0.5683, improve ≈ 1.56; soundness and contribution are both clipped to 10, overall 8.2, stored. Rounds 3 and 4 double and quadruple the training epochs, dropping loss to 1.0247 and 0.4568; both meet the threshold and are stored. The knowledge archive ends with these three ideas.

Rounds 5 and 6 are worth noting. The meta-Agent again proposes lowering the learning rate and adding L2 weight decay. Both ideas are already in `seen`, so novelty drops to 0.3, the reviewer marks them as duplicates, overall is only 1.0 and 1.3, and they do not pass. That is the deduplication mechanism: the same idea is reviewed only once, so the archive is not filled with repeats.

Compared with ADAS's closed loop, the structure is the same; only three settings take different values. ADAS mutates code, fitness is task accuracy, the population is a code archive. AI Scientist mutates research ideas, fitness is a paper-review score, the population is a knowledge archive.

The risk sits here as well. Reviewer and author are both AI; the closed loop has no external ground truth. Whether a review score of 8.2 corresponds to scientific value is not decided by anything outside the system. The paper already records cases where the AI wrote a negative result as an improvement—a hallucination. That structural issue is discussed in section 4.

This section treats a complete algorithm source file as an object that can evolve, and searches from it for better code. AI Scientist's variants are research ideas; AlphaEvolve turns variants back into code, but this time not the thin Agent function of section 1—an entire algorithm source file. The language model is the mutation operator: it reads the current program and outputs a code edit in a format called a diff. An automatic evaluator computes fitness. An evolutionary database maintains a population of different solutions under a "keep behavior as diverse as possible" rule. The three components match ADAS; the difference is scale: entire files, any language, and hour-scale parallel evaluation.

The paper's most representative result is matrix multiplication: the product of two 4×4 complex matrices with 48 scalar multiplications, the first improvement in 56 years since Strassen's 49 in 1969, found in about 15 mutations. Evolution also has a hard premise: fitness must be automatically evaluable. Each AlphaEvolve solution runs for several hours in the evaluator; problems that cannot be scored automatically cannot enter the loop. That is both its boundary and the reason it grows from FunSearch, which evolved a single function, to entire files. The next cell shows the carrier of the mutation operator: what a diff looks like.

AlphaEvolve reuses ADAS's three-piece kit, only swapping the variant from one Agent's code to an entire algorithm source file. Here we switch to genetics vocabulary, starting from the basics. In genetics, every trait of an organism is determined by a stretch of hereditary material, the **genome**; how the organism looks and acts is the result of running that genome, the **phenotype**. Mapped onto code: a piece of executable source is the genome, the program's runtime behavior is the phenotype, and the fitness computed by the evaluator decides whether it may keep reproducing.

Genome, mutation, selection, and population correspond in code space as follows:

- Genome: a piece of executable source. AlphaEvolve can evolve an entire file, in any language.
- Mutation: a language model reads the current program and outputs a code edit, expressed as a SEARCH/REPLACE diff: first the original code to replace, then what to replace it with.
- Fitness: the score produced by an automatic evaluator.
- Population: an evolutionary database that keeps a batch of behaviorally different solutions under a quality-diversity rule, rather than keeping only one optimum.

Mutation uses a diff rather than a full rewrite because a diff is a local edit. Most of a program is infrastructure; only a small block determines behavior. Editing only that block keeps the rest stable, so a mutation is more likely to succeed. A full rewrite, by contrast, easily breaks previously accumulated good structure. Treat code as hereditary material: mutation edits a small stretch of genes while the genome as a whole continues—the way biological evolution works.

The next cell demonstrates diff mutation on a bubble-sort program.

In [ ]:
# Mutation operator: parse and apply AlphaEvolve SEARCH/REPLACE diff blocks.
def apply_diff(old_code, search, replace):
    """Replace search text with replace in old_code; return the new code."""
    if search not in old_code:
        raise ValueError("SEARCH block not found in code")
    return old_code.replace(search, replace)

SRC = """def bubble(nums):
    for i in range(len(nums) - 1):
        for j in range(len(nums) - 1 - i):
            if nums[j] > nums[j + 1]:
                nums[j], nums[j + 1] = nums[j + 1], nums[j]
    return nums
"""

def eval_sort(src):
    """Run the sorting code; return the fraction correct on 3 arrays."""
    ns = {}
    exec(src, ns)
    fn = ns["bubble"]
    cases = [[3, 1, 2], [5, 4, 3, 2, 1], [1, 2, 3]]
    ok = 0
    for c in cases:
        ok += int(fn(list(c)) == sorted(c))
    return ok / len(cases)

print("original code accuracy:", eval_sort(SRC))

# Mutation 1: change ascending comparison to descending; sort direction reverses
search = "if nums[j] > nums[j + 1]:"
replace = "if nums[j] < nums[j + 1]:"
mut1 = apply_diff(SRC, search, replace)
print("mutation diff:")
print(f"<<<<<<< SEARCH\n{search}\n=======\n{replace}\n>>>>>>> REPLACE")
print("accuracy after mutation:", eval_sort(mut1))

# Mutation 2: outer loop scans one extra pass; behavior unchanged
mut2 = apply_diff(SRC, "range(len(nums) - 1)", "range(len(nums))")
print("accuracy after dropping outer -1:", eval_sort(mut2))

The contrast between the two mutations shows both sides of diff mutation. Original bubble sort is correct on all three cases; accuracy 1.0.

Mutation 1 changes the comparator from `>` to `<`, so sort direction flips from ascending to descending. Hand calculation on the first case [3, 1, 2]: sorting swaps smaller numbers backward, yielding [3, 2, 1], while the correct result is [1, 2, 3]—wrong. All three cases reverse; accuracy 0.0. Changing one symbol is enough for fitness to collapse. Code mutation is not as smooth as parameter mutation: edit the wrong place and everything fails.

Mutation 2 changes the outer loop from `range(len(nums) - 1)` to `range(len(nums))`, one extra pass. The last inner loop range is empty, so nothing happens, the result is unchanged, and accuracy is still 1.0. Some edits do not change the result and do not change fitness; that kind of mutation is a **neutral mutation**. It gives the population one more behaviorally identical variant. It does not change the score, but it can become material for a later combination.

The mutation operator's output is this SEARCH/REPLACE format: the SEARCH block is the original code to find, the REPLACE block is the new code, and replacement happens at the matched location. In real AlphaEvolve the language model generates this kind of diff; the evaluator runs only the program after replacement.

This subsection shows how selection improves a population as a whole when many individuals evolve together. A diff showed a single mutation; evolution also needs a population and selection: many individuals evolve at once, each generation the high-scoring individuals remain and are copied, and the low-scoring ones are removed. That procedure is a genetic algorithm. Below we run evolution on a small, analytically checkable target: a set of Fourier-basis coefficients approximating sin x + 0.5 cos 2x. The Fourier basis is a fixed set of functions; a weighted sum of those functions is fitted to the target, and the weights are the coefficient vector each individual evolves. Each individual is a 6-dimensional coefficient vector, mutation adds a small random perturbation to the coefficients, and selection keeps the two highest-scoring individuals as elites and copies them. The score is negative mean squared error, larger is better.

In [ ]:
# Target: approximate sin x + 0.5 cos 2x with Fourier-basis coefficients.
# An individual is 6 coefficients; the score is negative mean squared error, larger is better.
x = np.linspace(-np.pi, np.pi, 200)
y_target = np.sin(x) + 0.5 * np.cos(2 * x)

def basis_at(x):
    """Fourier basis: 1, x, sin x, cos x, sin 2x, cos 2x."""
    return np.stack([np.ones_like(x), x, np.sin(x), np.cos(x),
                     np.sin(2 * x), np.cos(2 * x)])

B = basis_at(x)

def fitness(coefs):
    """Approximation quality of a coefficient vector on the grid: negative mean squared error."""
    pred = B.T @ coefs
    return -float(np.mean((pred - y_target) ** 2))

coef_star, *_ = np.linalg.lstsq(B.T, y_target, rcond=None)
print("least-squares optimal score (reference):", round(fitness(coef_star), 4))

def evolve(pop, gen, elite=2, sigma=0.3, seed=0):
    """Evolve gen generations: each generation keep the best elite individuals and fill the population by mutated copies."""
    rng = np.random.RandomState(seed)
    curve = []
    for _ in range(gen):
        scores = np.array([fitness(p) for p in pop])
        curve.append(float(scores.max()))
        order = np.argsort(scores)[::-1][:elite]
        parents = [pop[i] for i in order]
        children = []
        while len(children) < len(pop) - elite:
            parent = parents[rng.randint(elite)]
            children.append(parent + sigma * rng.randn(len(parent)))
        pop = parents + children
    return curve, pop

rng0 = np.random.RandomState(1)
pop = [rng0.randn(6) * 2 for _ in range(30)]
curve, last_pop = evolve(pop, gen=60, elite=2, sigma=0.3, seed=2)
print("generation 0 best score:", round(curve[0], 4))
print("generation 60 best score:", round(curve[-1], 4))

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 3.5))
plt.plot(curve, color="#2c7fb8")
plt.xlabel("Generation")
plt.ylabel("Best fitness")
plt.title("Best fitness over generations")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Quality-diversity view: bin the final population by two behavior features; keep the highest score per bin
b0 = np.array([p[0] for p in last_pop])
b1 = np.array([p[1] for p in last_pop])
fs = np.array([fitness(p) for p in last_pop])
grid = np.full((20, 20), np.nan)
ix = np.clip(((b0 + 2.5) / 5.0 * 19).astype(int), 0, 19)
iy = np.clip(((b1 + 2.5) / 5.0 * 19).astype(int), 0, 19)
for a, b, f in zip(ix, iy, fs):
    grid[a, b] = f if np.isnan(grid[a, b]) else max(grid[a, b], f)

plt.figure(figsize=(5.5, 4.5))
im = plt.imshow(grid, origin="lower", cmap="viridis")
plt.colorbar(im, label="Fitness")
plt.xlabel("Behavior: coef[0]")
plt.ylabel("Behavior: coef[1]")
plt.title("Performance map of final population")
plt.tight_layout()
plt.show()
print("occupied behavior cells:", int(np.sum(~np.isnan(grid))), "/ 400")

The left plot first: best fitness rises with generation. Generation 0 is 30 random coefficient vectors, the best only −2.3756; by generation 60 it reaches −0.0072, close to the least-squares theoretical optimum −0.0. It can approach that value because the target sin x + 0.5 cos 2x is exactly a combination of the Fourier basis, which contains sin x and cos 2x, so some coefficient vector makes the error 0. Evolution does not need to know that; it only keeps high-scoring individuals and approaches step by step.

Walk through one generation. The population has 30 individuals, each a 6-dimensional coefficient vector:

1. Compute fitness. Negative mean squared error for each individual, 30 scores.
2. Select. Keep the 2 highest-scoring as elites; drop the other 28.
3. Mutate and copy. Draw 1 parent at random from the 2 elites, add Gaussian noise with σ = 0.3, and generate 28 children.
4. Fill. 2 elites plus 28 children equal 30, which enter the next generation.

Number of children = population size − number of elites = 30 − 2 = 28. Elites are kept and noise is added because the two jobs differ. Keeping elites is exploitation: the highest-scoring solution is inherited directly and does not regress. Adding noise is exploration: perturb near the optimum, looking for a possibly better neighbor. Missing either one, the process either stands still or degrades into random search. σ controls the exploration radius: larger σ takes coarser steps, smaller σ is finer; 0.3 balances the two in this example.

The right plot answers a different point: where the 30 individuals of the last generation sit. Take each individual's first two coefficients as behavior features, drop them into a 20×20 behavior grid, and keep only the highest score per cell. Only 16 cells are occupied; the population is not collapsed into one clump, but spread across different regions of behavior space. A solution may have no better neighbor nearby while a higher-scoring solution still exists elsewhere; that kind of solution is a local optimum. Keeping only one highest-scoring solution, the population easily stalls at the same local optimum. Quality diversity does not look only at who scores high, but also at who behaves differently, and keeps a batch of behaviorally different solutions. AlphaEvolve uses the same idea to maintain its evolutionary database.

The loop itself can work; risk concentrates on the evaluation function. This section shows what goes wrong when evolution scores its own outputs, and how to block it. The mutation operator only optimizes the score it is given. Once that score drifts from the true objective, evolution finds a shortcut: scores look high but the task is not really done. That behavior is called **reward hacking**. A demo exposes an evaluation hole: an Agent that memorizes the evaluation set's answers can get a perfect score. Then we split the evaluation set from the development set and watch how that guardrail blocks the shortcut.

Reward hacking means: evolution only optimizes the score it sees, not the true objective behind the score. Once the score can be inflated by a shortcut, and that shortcut does not actually complete the task, evolution will take the shortcut, because the shortcut scores higher.

A concrete example makes the mechanism clear. Suppose the 40 problems used to pick designs are fixed, from (11, 11) to (15, 18); call this batch the development set. An honest Agent uses the decompose_check recipe on every problem and scores 0.93. Now swap in another Agent that does not compute: it copies the 40 answers into a table and looks them up at evaluation time. All 40 development problems hit, score 1.00, higher than the honest Agent.

Looking only at the development-set score, evolution keeps the answer-copying Agent, because 1.00 is greater than 0.93. On a fresh batch of 40 unseen problems, from (20, 21) to (24, 28), the copying Agent finds no row and can only return a placeholder, all wrong, score 0.00. The honest Agent is still 0.93, because it did not learn an answer table; it learned a multiplication algorithm.

Hand calculation. Development set, 40 problems, copying gets all right, 40 / 40 = 1.00. Independent new problems, 40, no answers found, assume it returns "0", none correct, 0 / 40 = 0.00. The same Agent, two scores far apart. The issue is not how clever the Agent is; it is where the score comes from. When development problems and evaluation problems coincide, memorizing answers is the optimal strategy.

In [ ]:
# Reward-hacking demo: when development and evaluation share the same problems, hard-coded answers are a shortcut.
import numpy as np

A = np.repeat(np.arange(11, 16), 8)
B = np.tile(np.arange(11, 19), 5)
TRAIN = list(zip(A.tolist(), B.tolist()))
TRUE_TRAIN = [a * b for a, b in TRAIN]

H = np.repeat(np.arange(20, 25), 8)
K = np.tile(np.arange(21, 29), 5)
HOLD = list(zip(H.tolist(), K.tolist()))
TRUE_HOLD = [a * b for a, b in HOLD]

def honest_agent(task):
    """Honest Agent: solve with the framework's step-by-step recipe."""
    a, b = task
    return solve(a, b, "decompose_check")

answer_table = {(a, b): str(a * b) for (a, b) in TRAIN}

def cheat_agent(task):
    """Shortcut Agent: hard-code development answers; answer 0 on new problems."""
    return answer_table.get(task, "0")

print("honest Agent  development accuracy:  ", round(eval_agent(honest_agent, TRAIN, TRUE_TRAIN), 2))
print("honest Agent  held-out accuracy:     ", round(eval_agent(honest_agent, HOLD, TRUE_HOLD), 2))
print("shortcut Agent  development accuracy:", round(eval_agent(cheat_agent, TRAIN, TRUE_TRAIN), 2))
print("shortcut Agent  held-out accuracy:   ", round(eval_agent(cheat_agent, HOLD, TRUE_HOLD), 2))

The output lays the problem out clearly. The honest Agent scores 0.93 on both the development set and the independent new problems: the algorithm it learned does not depend on specific problems, so a new batch still works. The shortcut Agent scores 1.0 on the development set, the highest of the four numbers, and 0.0 on the independent new problems, the lowest of the four.

If this section evaluated only the development set, evolution would keep the shortcut Agent, because it has the highest score. It never learned multiplication; it only memorized a table of answers. The true objective (solve any two-digit multiplication) is not met at all. That is reward hacking: the score is optimized to the maximum, the true objective does not improve.

A common misunderstanding needs a correction. The shortcut Agent is not cheating; it did exactly what it was asked to do—score high on the given evaluation. Evolution has no intent; it only amplifies high-scoring designs. If the score is defined wrongly, the direction of optimization is wrong. That is not the Agent's fault; it is a fault of evaluation design. The only repair is to take the score from a place the shortcut cannot reach.

In [ ]:
# Repair: split development set from evaluation set; the evaluation set never participates in selection.
# Pick a recipe on the development set, then report true performance on an independent evaluation set.
def run_search_with_holdout(dev, hold, rounds=4):
    """Search on the development set, report on an independent evaluation set. Return (best recipe on dev, dev accuracy, held-out accuracy)."""
    dev_problems, dev_true = dev
    hold_problems, hold_true = hold
    pool = ["decompose_check", "ensemble", "direct", "decompose_check"]
    best_recipe, best_acc = None, -1.0
    for it in range(rounds):
        recipe = pool[it % len(pool)]
        acc = eval_agent(compile_agent(wrap_recipe(recipe)),
                         dev_problems, dev_true)
        if acc > best_acc:
            best_recipe, best_acc = recipe, acc
    hold_acc = eval_agent(compile_agent(wrap_recipe(best_recipe)),
                          hold_problems, hold_true)
    return best_recipe, best_acc, hold_acc

recipe, dev_acc, hold_acc = run_search_with_holdout((TRAIN, TRUE_TRAIN),
                                                     (HOLD, TRUE_HOLD))
print("recipe selected on development set:", recipe, "  development accuracy:", round(dev_acc, 2))
print("independent evaluation accuracy:", round(hold_acc, 2))
print("Conclusion: hard-coded answers score 0 on independent evaluation; evaluation independence blocks the shortcut.")

The repair is to split the development set from the evaluation set completely. The development set participates in selection: run each recipe on it and pick the highest accuracy. The evaluation set never participates in selection; it only reports how the finally chosen design does on unseen problems. A hard-coded-answer Agent scores 1.0 on the development set and would be selected, but when the evaluation set reports, it finds no row, scores 0.00, and is blocked by this guardrail.

In the output, the development set selected the ensemble recipe, development 1.0, independent evaluation also 1.0. Ensemble can pass because it is not memory: it runs several recipes and votes, so problems absent from the development set are still computed correctly. A high development score together with a high evaluation score means this score really learned something.

Behind this is the oldest split in machine learning: the training set is used to tune parameters, the test set is used to report true performance. In Agent evolution the development set corresponds to the training set, the independent evaluation set to the test set. Three principles to keep: the evaluation set never participates in selection; evaluation problems and development problems have the same distribution but do not repeat; reported scores use only the evaluation set. Hold these three, and reward hacking has nowhere to hide.

The risk is not only reward hacking. In AI Scientist, reviewer and author are both AI, so the closed loop lacks external ground truth. The paper already records hallucinations such as writing a negative result as an improvement, or claiming to have used the wrong hardware. When AI authors submit in bulk, the reviewing system is also overloaded.

Each of the three papers gives an engineering guardrail: ADAS recommends executing generated code in a container, AI Scientist recommends sandboxing and labeling AI output, AlphaEvolve acknowledges that evaluability itself is a premise. Taken together: open-ended evolution can take an Agent past the ceiling of human design, and it can also move in an uncontrolled direction. Independent evaluation is the only guardrail.

## Summary

This section is organized around letting an Agent improve itself. What we covered:

- [ ] Three elements of open-ended evolution: mutation operator, selection mechanism, population and archive
- [ ] Minimal mutation demo: change one word in a prompt, and the score changes
- [ ] Parameter-space search: hill-climbing and random-search traces on a deterministic scoring function
- [ ] ADAS minimal closed loop: meta-Agent writes forward code → compile → evaluate → store
- [ ] Stepping stones: later Agents combine earlier Agents' components; the archive is memory for the mutation operator
- [ ] AI Scientist scales the loop to research: idea → experiment → review; fitness is paper review
- [ ] AlphaEvolve takes an entire file as genome, a diff as mutation operator, and an evaluator as fitness
- [ ] Failure mode: reward hacking comes from evaluation holes; evaluation independence is the only guardrail

## Exercises

> You may ask an AI to explain the idea. Do not ask it to finish the exercise for you.

Each of the three problems has a blank. Reference answers are already filled in. Compute them by hand first, then run the asserts to check.

**Exercise 1: Complete fitness and selection**

Given `scores = {"a": 0.3, "b": 0.9, "c": 0.6, "d": 0.4}`, complete `select_top_k(scores, k=2)` so that it returns the 2 keys with the highest scores, from high to low; then complete `archive_update(archive, name, score)` so that it stores only when score exceeds the archive's current minimum.

Hint: sort with `sorted(scores.items(), key=lambda kv: kv[1], reverse=True)` and slice; before storing, compare with `min(archive.values())`.

In [ ]:
def select_top_k(scores, k=2):
    """Return the k keys with the highest scores, from high to low."""
    ordered = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
    return [name for name, _ in ordered[:k]]   # blank goes here

def archive_update(archive, name, score):
    """Store when score exceeds the archive's current minimum; return whether it was stored."""
    if not archive:
        archive[name] = score
        return True
    if score > min(archive.values()):   # blank goes here
        archive[name] = score
        return True
    return False

scores = {"a": 0.3, "b": 0.9, "c": 0.6, "d": 0.4}
assert select_top_k(scores, k=2) == ["b", "c"], "top-2 should be b and c"

arch = {"x": 0.5}
assert archive_update(arch, "y", 0.9) is True
assert archive_update(arch, "z", 0.4) is False
assert "z" not in arch
print("Takeaway: select top-k by score; store only when beating the current minimum.")

**Exercise 2: Complete compiling and calling Agent code**

The meta-Agent returns a source-code string. Complete `compile_agent(src)`: use `exec` to compile the source into a namespace, take out `forward`, and return it; then complete `run_agent(fn, task)` so that it calls the function and returns the answer. The given source already defines `forward(task)`.

Hint: after `exec(src, ns)`, `ns["forward"]` is the callable.

In [ ]:
def compile_agent(src):
    """Compile a forward source-code string into a callable function."""
    ns = {}
    exec(src, ns)
    return ns["forward"]   # blank goes here

def run_agent(fn, task):
    """Call the Agent on one task; return the answer string."""
    return fn(task)   # blank goes here

src = """def forward(task):
    a, b = task
    return str(a * b)
"""
agent = compile_agent(src)
assert callable(agent), "the compiled result should be callable"
assert run_agent(agent, (7, 8)) == "56"
print("Takeaway: source generated by the meta-Agent can be exec'd into a callable and run.")

**Exercise 3: Complete the review-score threshold decision**

Review returns a score dict. Complete `should_accept(review, threshold=6)`: accept when `overall` meets the threshold and `contribution` is at least 3, otherwise reject; then complete `merge_reviews(reviews)` so that it averages `overall` across several reviews.

Hint: read `overall` and `contribution` by key; average with `sum(...) / len(...)`.

In [ ]:
def should_accept(review, threshold=6):
    """Accept when overall meets the threshold and contribution is at least 3."""
    return review["overall"] >= threshold and review["contribution"] >= 3   # blank goes here

def merge_reviews(reviews):
    """Average overall across several reviews."""
    return sum(r["overall"] for r in reviews) / len(reviews)   # blank goes here

rev = {"soundness": 5, "presentation": 4, "contribution": 6, "overall": 6}
assert should_accept(rev) is True, "overall 6 and contribution 6 should be accepted"
assert should_accept({"overall": 6, "contribution": 2}) is False
assert merge_reviews([{"overall": 5}, {"overall": 7}]) == 6.0
print("Takeaway: review threshold decisions and averaging across reviewers.")

## References

- Hu et al., [Automated Design of Agentic Systems](https://arxiv.org/abs/2408.08435), 2024 — a meta-Agent designs Agents in code space; the first of this lecture's three levels. Code https://github.com/ShengranHu/ADAS
- Lu et al., [The AI Scientist: Towards Fully Automated Open-Ended Scientific Discovery](https://arxiv.org/abs/2408.06292), 2024 — a fully automatic research loop of idea → experiment → paper → review. Code https://github.com/SakanaAI/AI-Scientist
- Novikov et al., [AlphaEvolve: A coding agent for scientific and algorithmic discovery](https://arxiv.org/abs/2506.13131), 2025 — an evolutionary coding Agent with code as genome and an evaluator as fitness
- Romera-Paredes et al., [FunSearch: Mathematical discoveries from program search with LLMs](https://arxiv.org/abs/2312.02174), 2023 — AlphaEvolve's predecessor; an LLM evolves a single function
- Wang et al., [Quality-Diversity algorithms: A generic definition and an illustration](https://arxiv.org/abs/2103.04313), 2021 — MAP-Elites and quality diversity, the algorithmic source of AlphaEvolve's population
- Clune, [AI-Generating Algorithms: An Alternate Paradigm](https://arxiv.org/abs/1901.01346), 2019 — theoretical origin of this lecture; the three pillars of AI-GA
- This repo's `llm_client.py` — unified entry for all LLM demos; the scripted path keeps the notebook executable offline